# no-grad-context-mgr-update — worked example 2: nested NoGrad restores the outer value

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `no-grad-context-mgr-update`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

Because `NoGrad` restores the previous value (not a hard-coded `True`), nesting two `NoGrad` blocks keeps grad disabled until the OUTERMOST block exits. Each `__exit__` returns to exactly the state seen on its `__enter__`.

## Worked solution

We reuse the save-restore `NoGrad`. The point of this example is the nested case: entering an outer `NoGrad` sets the flag to `False` (saving the outer `True`), and entering an inner `NoGrad` saves that `False` and sets `False` again. When the inner block exits, it restores `False` (the saved previous value) rather than `True`, so grad stays disabled inside the still-open outer block. Only when the outer block exits does the flag return to `True`. We print the flag at the deepest point and after each exit to confirm the staircase of restorations.

In [ ]:
grad_tracking_enabled = True

class NoGrad:
    def __enter__(self):
        global grad_tracking_enabled
        self._prev = grad_tracking_enabled
        grad_tracking_enabled = False
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        global grad_tracking_enabled
        grad_tracking_enabled = self._prev

with NoGrad():
    with NoGrad():
        print('deep:', grad_tracking_enabled)      # False
    print('after inner:', grad_tracking_enabled)   # still False
print('after outer:', grad_tracking_enabled)        # True